# Huấn luyện Super Mario Bros PPO trên Google Colab (`RL_Mario_Solving`)

Notebook này tự động hóa trọn gói quy trình huấn luyện, theo dõi chẩn đoán, đánh giá chuẩn đề bài (`seed=2026`, 20 lượt) và **đóng gói tải kết quả về đúng cấu trúc thư mục của project `RL_Mario_Solving`** (`runs/` và `results/`).

### Checklist trước khi bấm chạy
1. Trên thanh menu Colab chọn **Runtime $\rightarrow$ Change runtime type $\rightarrow$ Hardware accelerator: `T4 GPU`** (hoặc `L4`/`A100` nếu có Colab Pro).
2. Chạy tuần tự từ **Bước 1 $\rightarrow$ Bước 8**.
3. Khuyên bật `SAVE_TO_DRIVE = True` để mọi `progress.csv`, `evals.csv`, `checkpoints/*.pt` và `videos/*.mp4` được ghi trực tiếp lên Google Drive — nếu Colab bị ngắt kết nối giữa chừng chỉ cần bật `AUTO_RESUME = True` để chạy tiếp đúng bước đang dừng.
4. Sau khi tải file `.zip` ở **Bước 8**, giải nén thẳng vào thư mục gốc `D:\University_Subject\RL\RL_Mario_Solving` trên máy tính là toàn bộ `runs/` và `results/` sẽ tự động nằm đúng vị trí của project.

## Bước 1. Kiểm tra phần cứng Colab & Gắn Google Drive

In [ ]:
#@title 1.1 Kiểm tra CPU, RAM, GPU, phiên bản Python & gắn Google Drive
import os
import platform
import shutil
import subprocess
import sys

SAVE_TO_DRIVE = True  #@param {type:"boolean"}
DRIVE_ROOT_DIR = "/content/drive/MyDrive/RL_Mario_Solving_Runs"  #@param {type:"string"}

print(f"Python   : {sys.version.split()[0]} ({platform.platform()})")
print(f"CPU cores: {os.cpu_count()}")
try:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / (1024 ** 3)
    print(f"RAM      : {ram_gb:.2f} GB")
except Exception:
    pass

gpu_out = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if gpu_out.returncode == 0 and gpu_out.stdout.strip():
    print(f"GPU      : {gpu_out.stdout.strip()}")
else:
    print("GPU      : Không phát hiện GPU (đang chạy CPU). Khuyên bật T4 GPU trong Runtime -> Change runtime type.")

if sys.version_info >= (3, 13):
    print(
        "[CẢNH BÁO] Colab đang dùng Python 3.13+, trong khi pyproject.toml yêu cầu numpy<2 (Python 3.10–3.12). "
        "Nếu bước cài đặt báo lỗi numpy<2, hãy chuyển Runtime -> Fallback runtime hoặc dùng conda/miniforge Python 3.11."
    )

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(DRIVE_ROOT_DIR, exist_ok=True)
    print(f"Google Drive đã sẵn sàng tại: {DRIVE_ROOT_DIR}")
else:
    print("Đang lưu cục bộ tại /content (lưu ý dữ liệu sẽ mất khi Colab reset runtime).")

## Bước 2. Clone mã nguồn & Cài đặt môi trường `mario-ppo`

In [ ]:
#@title 2.1 Clone / cập nhật repo GitHub & cài đặt phụ thuộc từ pyproject.toml
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Eternity-KT/RL_Mario_Solving.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
PROJECT_DIR = Path("/content/RL_Mario_Solving")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", BRANCH], check=True)

os.chdir(PROJECT_DIR)
print("Thư mục làm việc hiện tại:", Path.cwd())
subprocess.run(["git", "log", "--oneline", "-n", "3"], check=True)

print("\n==> Cài đặt công cụ hệ thống (build-essential, ffmpeg)...")
subprocess.run("apt-get update -qq && apt-get install -y -qq build-essential ffmpeg libgl1", shell=True, check=True)

print("\n==> Cài đặt mario-ppo (`pip install -e .`) và matplotlib...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)

print("\n==> Kiểm tra phiên bản các thư viện cốt lõi trong tiến trình mới:")
verify_code = (
    "import torch, numpy, gymnasium, mo_gymnasium, nes_py, gym_super_mario_bros, mario_ppo; "
    "print(f'torch={torch.__version__} | cuda={torch.cuda.is_available()} | ' "
    "f'numpy={numpy.__version__} | gymnasium={gymnasium.__version__}')"
)
subprocess.run([sys.executable, "-c", verify_code], check=True)

## Bước 3. Kiểm thử nhanh (Sanity Check) trước khi huấn luyện

In [ ]:
#@title 3.1 Chạy pytest xác nhận môi trường & mã nguồn đạt chuẩn
import subprocess
import sys

RUN_FULL_PYTEST = False  #@param {type:"boolean"}

if RUN_FULL_PYTEST:
    cmd = [sys.executable, "-m", "pytest", "-q"]
else:
    cmd = [sys.executable, "-m", "pytest", "-q", "tests/test_model.py", "tests/test_wrappers.py", "tests/test_gae.py", "tests/test_ppo.py"]

print("Chạy:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Tất cả bài kiểm thử đã PASS!")

## Bước 4. Cấu hình tham số huấn luyện

Gợi ý cấu hình theo từng nhiệm vụ đồ án:
- **Màn 1-1 Baseline (5M bước):** `LEVEL="1-1"`, `RUN_NAME="run1_colab"`, `TOTAL_STEPS=5000000`, `GAMMA=0.9`, `MAX_NO_PROGRESS_STEPS=0`
- **Màn 1-1 chống kẹt ống nước (5M bước):** đặt `MAX_NO_PROGRESS_STEPS=200`
- **Màn thưởng 1-2 Transfer Learning (3.5M–5M bước):** `LEVEL="1-2"`, `RUN_NAME="transfer_1_2"`, điền đường dẫn checkpoint 1-1 vào `TRANSFER_FROM_CKPT`
- **Thí nghiệm đối chứng $\gamma = 0.99$ (2M bước):** `LEVEL="1-1"`, `RUN_NAME="ablation_gamma_099"`, `TOTAL_STEPS=2000000`, `GAMMA=0.99`, `SEED=1`

> **Lưu ý hiệu năng Colab:** Colab Free T4 thường có **2 vCPU**, nên `NUM_ENVS = 8` là mức tối ưu giữa tốc độ thu dữ liệu và RAM (nếu bạn dùng Colab Pro $\ge 8$ vCPU hãy nâng `NUM_ENVS = 16`). Đặt `NUM_STEPS = 512` khi `NUM_ENVS = 8` sẽ giữ nguyên kích thước lô chuẩn `8 × 512 = 4096` mẫu/lần cập nhật!

In [ ]:
#@title 4.1 Thiết lập cấu hình lượt chạy (Run Config)
import os
from pathlib import Path

LEVEL = "1-1"  #@param ["1-1", "1-2", "1-4", "2-1", "3-1", "3-3", "4-1", "5-2", "6-1"]
RUN_NAME = "run1_colab"  #@param {type:"string"}
TOTAL_STEPS = 5000000  #@param {type:"integer"}
SEED = 1  #@param {type:"integer"}

# Tự điều chỉnh song song môi trường: 0 = tự động (8 envs cho <=4 vCPU, 16 envs cho >=8 vCPU)
NUM_ENVS = 0  #@param {type:"integer"}
NUM_STEPS = 0  #@param {type:"integer"}  # 0 = tự động giữ lô 4096 mẫu (4096 // NUM_ENVS)
TORCH_THREADS = 2  #@param {type:"integer"}

# Siêu tham số PPO & mẹo huấn luyện (Mục 3.4 đề bài)
GAMMA = 0.9  #@param {type:"number"}
LR = 0.00025  #@param {type:"number"}
ENT_COEF = 0.01  #@param {type:"number"}
MAX_NO_PROGRESS_STEPS = 200  #@param {type:"integer"}  # 0 = tắt, 200 = bật NoProgressTruncation
TARGET_KL = ""  #@param {type:"string"}  # Để trống = tắt (null), hoặc nhập "0.02"

# Lưu checkpoint & tự động khôi phục khi Colab rớt mạng
CHECKPOINT_EVERY = 500000  #@param {type:"integer"}
EVAL_EPISODES = 5  #@param {type:"integer"}
RECORD_VIDEO = True  #@param {type:"boolean"}
AUTO_RESUME = True  #@param {type:"boolean"}

# (Chỉ điền khi chạy Transfer Learning sang màn thưởng, ví dụ đường dẫn tới step_003500000.pt của màn 1-1)
TRANSFER_FROM_CKPT = ""  #@param {type:"string"}

cpu_n = os.cpu_count() or 2
if NUM_ENVS <= 0:
    NUM_ENVS = 16 if cpu_n >= 8 else 8
if NUM_STEPS <= 0:
    NUM_STEPS = max(128, 4096 // NUM_ENVS)

BASE_RUNS_ROOT = Path(DRIVE_ROOT_DIR) / "runs" if SAVE_TO_DRIVE else Path("/content/RL_Mario_Solving/runs")
RUN_DIR = BASE_RUNS_ROOT / LEVEL / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

OUT_RESULTS_DIR = (
    (Path(DRIVE_ROOT_DIR) / "results" / f"man_{LEVEL.replace('-', '_')}")
    if SAVE_TO_DRIVE
    else (Path("/content/RL_Mario_Solving/results") / f"man_{LEVEL.replace('-', '_')}")
)
OUT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("=== CẤU HÌNH HUẤN LUYỆN ===")
print(f"Màn chơi          : {LEVEL}")
print(f"Tên run           : {RUN_NAME}")
print(f"Thư mục lưu run   : {RUN_DIR}")
print(f"Thư mục kết quả   : {OUT_RESULTS_DIR}")
print(f"Song song         : num_envs={NUM_ENVS}, num_steps={NUM_STEPS} (batch={NUM_ENVS * NUM_STEPS}), torch_threads={TORCH_THREADS}")
print(f"Tổng bước         : {TOTAL_STEPS:,} | checkpoint mỗi {CHECKPOINT_EVERY:,} bước")
print(f"Siêu tham số      : gamma={GAMMA}, lr={LR}, ent_coef={ENT_COEF}, max_no_progress_steps={MAX_NO_PROGRESS_STEPS}, target_kl={TARGET_KL or 'null'}")

## Bước 5. Khởi chạy huấn luyện PPO (hỗ trợ tự động Resume)

In [ ]:
#@title 5.1 Chạy `python -m mario_ppo.train` & ghi log trực tiếp
import os
import subprocess
import sys
from pathlib import Path

os.chdir("/content/RL_Mario_Solving")

resume_ckpt = ""
existing_ckpts = sorted((RUN_DIR / "checkpoints").glob("step_*.pt")) if (RUN_DIR / "checkpoints").exists() else []
if AUTO_RESUME and existing_ckpts:
    resume_ckpt = str(existing_ckpts[-1])
    print(f"[AUTO-RESUME] Tìm thấy checkpoint gần nhất, tiếp tục huấn luyện từ: {resume_ckpt}")
elif TRANSFER_FROM_CKPT.strip():
    ckpt_path = Path(TRANSFER_FROM_CKPT.strip())
    assert ckpt_path.is_file(), f"Không tìm thấy checkpoint transfer: {ckpt_path}"
    resume_ckpt = str(ckpt_path)
    print(f"[TRANSFER] Khởi tạo trọng số từ checkpoint: {resume_ckpt}")
else:
    print("[NEW RUN] Bắt đầu huấn luyện mới từ đầu.")

cmd = [
    sys.executable, "-m", "mario_ppo.train",
    "--config", "configs/default.yaml",
    f"level={LEVEL}",
    f"seed={SEED}",
    f"run_dir={RUN_DIR}",
    f"torch_threads={TORCH_THREADS}",
    f"checkpoint_every={CHECKPOINT_EVERY}",
    f"eval_episodes={EVAL_EPISODES}",
    f"video={'true' if RECORD_VIDEO else 'false'}",
    f"env.max_no_progress_steps={MAX_NO_PROGRESS_STEPS}",
    f"algo.num_envs={NUM_ENVS}",
    f"algo.num_steps={NUM_STEPS}",
    f"algo.total_steps={TOTAL_STEPS}",
    f"algo.gamma={GAMMA}",
    f"algo.lr={LR}",
    f"algo.ent_coef={ENT_COEF}",
]
if TARGET_KL.strip():
    cmd.append(f"algo.target_kl={TARGET_KL.strip()}")
if resume_ckpt:
    cmd.append(f"resume={resume_ckpt}")

print("Lệnh thực thi:")
print(" ", " ".join(cmd))
print("-" * 80)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
log_path = RUN_DIR / "train.log"

with log_path.open("a", encoding="utf-8") as log_f:
    log_f.write("\n=== START: " + " ".join(cmd) + " ===\n")
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        # Lọc bớt cảnh báo cũ lặp lại của gym để output Colab gọn gàng
        if "Gym has been unmaintained" in line or "gymnasium.farama.org" in line or "replace 'import gym'" in line or "Please upgrade to Gymnasium" in line:
            continue
        print(line, end="")
        log_f.write(line)
        log_f.flush()
    ret_code = proc.wait()

print("-" * 80)
print(f"Kết thúc huấn luyện với mã thoát: {ret_code}")
if ret_code != 0:
    raise RuntimeError(f"Quá trình huấn luyện dừng với mã lỗi {ret_code}. Kiểm tra {log_path}")

## Bước 6. Chẩn đoán tự động & Xuất 6 biểu đồ đường cong học (300 DPI)

In [ ]:
#@title 6.1 Chạy `scripts/plot_nguoi2_metrics.py` & hiển thị ảnh lưới 2x3 ngay trên Colab
import subprocess
import sys
from IPython.display import Image, display

cmd = [
    sys.executable,
    "scripts/plot_nguoi2_metrics.py",
    "--run-dir", str(RUN_DIR),
    "--out-dir", str(OUT_RESULTS_DIR),
    "--dpi", "300",
]
subprocess.run(cmd, check=True)

panel_img = OUT_RESULTS_DIR / "learning_curves_6_panel.png"
if panel_img.is_file():
    display(Image(filename=str(panel_img), width=1000))

## Bước 7. Đánh giá chính thức theo giao thức đề bài (`seed=2026`, 20 lượt)

In [ ]:
#@title 7.1 Tự chọn checkpoint tốt nhất, chạy `mario_ppo.evaluate` & xem video vượt màn
import base64
import csv
import json
import subprocess
import sys
from pathlib import Path
from IPython.display import HTML, display

EVAL_CKPT_OVERRIDE = ""  #@param {type:"string"}  # Để trống = tự chọn checkpoint có flag_rate cao nhất trong evals.csv
EVAL_SEED = 2026  #@param {type:"integer"}
EVAL_NUM_EPISODES = 20  #@param {type:"integer"}

def pick_best_checkpoint(run_dir: Path) -> Path:
    ckpt_dir = run_dir / "checkpoints"
    all_ckpts = sorted(ckpt_dir.glob("step_*.pt"))
    if not all_ckpts:
        raise FileNotFoundError(f"Không tìm thấy checkpoint nào trong {ckpt_dir}")

    evals_csv = run_dir / "evals.csv"
    if evals_csv.is_file():
        best_key = None
        best_step = None
        with evals_csv.open("r", encoding="utf-8", newline="") as f:
            for row in csv.DictReader(f):
                step = int(float(row["global_step"]))
                fr = float(row["flag_rate"])
                xs = [float(x) for x in row["x_pos"].split()] if row.get("x_pos") else [0.0]
                mean_x = sum(xs) / max(1, len(xs))
                cand = ckpt_dir / f"step_{step:09d}.pt"
                if cand.is_file():
                    key = (fr, mean_x, step)
                    if best_key is None or key > best_key:
                        best_key = key
                        best_step = step
        if best_step is not None:
            print(f"Chọn checkpoint tốt nhất theo evals.csv: step_{best_step:09d}.pt (flag_rate={best_key[0]:.2f}, mean_x={best_key[1]:.1f})")
            return ckpt_dir / f"step_{best_step:09d}.pt"

    print(f"Chọn checkpoint mới nhất: {all_ckpts[-1].name}")
    return all_ckpts[-1]

BEST_CKPT = Path(EVAL_CKPT_OVERRIDE.strip()) if EVAL_CKPT_OVERRIDE.strip() else pick_best_checkpoint(RUN_DIR)
JSON_OUT = OUT_RESULTS_DIR / f"ket-qua-{LEVEL}.json"
VIDEO_OUT = OUT_RESULTS_DIR / f"{LEVEL}.mp4"

eval_cmd = [
    sys.executable, "-m", "mario_ppo.evaluate",
    "--ckpt", str(BEST_CKPT),
    "--level", LEVEL,
    "--episodes", str(EVAL_NUM_EPISODES),
    "--seed", str(EVAL_SEED),
    "--out", str(JSON_OUT),
    "--video", str(VIDEO_OUT),
]
print("Chạy đánh giá chuẩn:", " ".join(eval_cmd))
subprocess.run(eval_cmd, check=True)

res = json.loads(JSON_OUT.read_text(encoding="utf-8"))
print("\n=== TÓM TẮT KẾT QUẢ CHẤM ĐIỂM CHÍNH THỨC ===")
print(f"Màn           : {res['level']}")
print(f"Checkpoint    : {BEST_CKPT.name} (step {res['global_step']:,})")
print(f"flag_rate     : {res['flag_rate']:.2f} ({int(round(res['flag_rate'] * res['episodes']))}/{res['episodes']} lượt chạm cờ)")
print(f"mean_x_pos    : {res['mean_x_pos']:.1f}")
print(f"Trạng thái    : {'VƯỢT MÀN (ĐẠT >= 0.8)' if res['solved'] else 'CHƯA VƯỢT MÀN (< 0.8)'}")

if VIDEO_OUT.is_file():
    b64 = base64.b64encode(VIDEO_OUT.read_bytes()).decode("ascii")
    display(HTML(f'<video width="512" controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

## Bước 8. Đóng gói & Tải kết quả về đúng cấu trúc thư mục project `RL_Mario_Solving`

Cell dưới đây đóng gói toàn bộ `runs/<LEVEL>/<RUN_NAME>/` và `results/man_<LEVEL>/` thành **1 tệp `.zip` giữ nguyên đường dẫn tương đối của project**, lưu bản sao trên Google Drive và tự động tải xuống máy tính của bạn.

### Cách đưa vào đúng thư mục project trên máy Windows của bạn (1 lệnh duy nhất):
Sau khi trình duyệt tải xong file `mario_ppo_<LEVEL>_<RUN_NAME>.zip` (ví dụ trong `Downloads`), mở PowerShell tại `D:\University_Subject\RL\RL_Mario_Solving` và chạy:
```powershell
Expand-Archive -Path "$HOME\Downloads\mario_ppo_1-1_run1_colab.zip" -DestinationPath . -Force
```
Toàn bộ `runs/...` và `results/...` sẽ tự động khớp vào đúng vị trí thư mục của project!

In [ ]:
#@title 8.1 Nén theo đúng cấu trúc `runs/` + `results/` và tải về máy
import zipfile
from pathlib import Path

INCLUDE_ALL_CHECKPOINTS = False  #@param {type:"boolean"}  # False = chỉ đóng gói checkpoint tốt nhất + mới nhất cho nhẹ (~40MB); True = đóng gói tất cả
AUTO_DOWNLOAD_BROWSER = True  #@param {type:"boolean"}

zip_name = f"mario_ppo_{LEVEL}_{RUN_NAME}.zip"
zip_dir = Path(DRIVE_ROOT_DIR) / "exports" if SAVE_TO_DRIVE else Path("/content")
zip_dir.mkdir(parents=True, exist_ok=True)
zip_path = zip_dir / zip_name

all_ckpts = sorted((RUN_DIR / "checkpoints").glob("step_*.pt")) if (RUN_DIR / "checkpoints").exists() else []
keep_ckpts = set(all_ckpts) if INCLUDE_ALL_CHECKPOINTS else set()
if not INCLUDE_ALL_CHECKPOINTS:
    if "BEST_CKPT" in globals() and Path(BEST_CKPT).is_file():
        keep_ckpts.add(Path(BEST_CKPT))
    if all_ckpts:
        keep_ckpts.add(all_ckpts[-1])

added = []
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    # 1. Đóng gói runs/<LEVEL>/<RUN_NAME>/...
    run_arc_root = Path("runs") / LEVEL / RUN_NAME
    for p in sorted(RUN_DIR.rglob("*")):
        if not p.is_file():
            continue
        rel = p.relative_to(RUN_DIR)
        if rel.parts[0] == "checkpoints" and p not in keep_ckpts:
            continue
        arc = (run_arc_root / rel).as_posix()
        zf.write(p, arc)
        added.append(arc)

    # 2. Đóng gói results/man_<LEVEL>/...
    res_arc_root = Path("results") / f"man_{LEVEL.replace('-', '_')}"
    for p in sorted(OUT_RESULTS_DIR.rglob("*")):
        if not p.is_file():
            continue
        rel = p.relative_to(OUT_RESULTS_DIR)
        arc = (res_arc_root / rel).as_posix()
        zf.write(p, arc)
        added.append(arc)

size_mb = zip_path.stat().st_size / (1024 ** 2)
print(f"Đã tạo gói kết quả: {zip_path} ({size_mb:.2f} MB, {len(added)} tệp)")
for item in added:
    print("  +", item)

print("\nLệnh giải nén vào đúng thư mục gốc project trên Windows PowerShell:")
print(f'  Expand-Archive -Path "$HOME\\Downloads\\{zip_name}" -DestinationPath "D:\\University_Subject\\RL\\RL_Mario_Solving" -Force')

if AUTO_DOWNLOAD_BROWSER:
    from google.colab import files
    files.download(str(zip_path))